<a href="https://colab.research.google.com/github/Dibyendu-Dhauri/youtube-video-qa-assistant/blob/main/YouTube_Video_Q%26A_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install libraries

In [ ]:
!pip install -q youtube-transcript-api langchain-community \
faiss-cpu tiktoken python-dotenv  langchain-text-splitters transformers accelerate torch langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings


/tmp/ipykernel_2107/71919881.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [ ]:

# only the ID, not full URL
# video_id = "Gfr50f6ZBvo"
video_id = 'kovF86_WqmQ'
try:

    ytt_api = YouTubeTranscriptApi()

    # If you don’t care which language, this returns the “best” one
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Now India has heat waves. Heat waves. Heat waves. Heat waves. India is in the grip of an intense and widespread heat wave in Maharashtra's Nagpur. At least 26 people have been reported dead. The heat has been more prolonged and over larger parts of the country. >> Is this being considered as as a sort of a national emergency at all? >> It's the peak of the summer of 2026. You're in your living room sweating profusely and waiting for the power to return so that your air conditioner can save you from an imminent heat stroke. But outside the window, out on the streets, there's people trying to make ends meet in that same scorching heat. The delivery riders, the street vendors, and those without a roof, all burning. Even birds dehydrated are starting to drop dead. And all of this is happening because of one single phenomena. >> Felino. >> El Nino. >> El Nino. >> The world has officially entered an El Nino phase. >> It can mean devastating floods, extreme droughts, intense heat waves, or ra

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

## Step 3 - Augmentation

In [ ]:
# pipe = pipeline(
#     "text-generation",
#     model="microsoft/Phi-3-mini-4k-instruct",
#     device_map="auto"
# )
# pipe = pipeline(
#     "text-generation",
#     model="google/gemma-2-2b-it",
#     device_map="auto"
# )
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=64,
    device_map="auto"
)

llm = HuggingFacePipeline(pipeline=pipe)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      it for electricity. Again, the effects of El Nino are disproportional. It's not you and me that are affected. It's the ones who would never come across this videos. Our labor workforce that are going to take the brunt of it all. The construction workers, the ricksha drivers, the street sellers. None of them can step out in the peak cards of heat. An entire start of economy loses millions of working hours to this heat. You know in 2024 heat cost India an estimated 247 billion potential working hours and 194 billion in lost income and is projected to get worse. India is on track to lose the equivalent of 34 million full-time jobs to heat stress more than any country on earth. You know in rural areas the drought pushes people toward dirtier water sources which means we see a spike in waterborn diseases like deni kera typhoi